In [1]:
# 라이브러리 호출
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings
import platform

warnings.filterwarnings('ignore')

# 운영체제별 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False

In [5]:
# 데이터 프레임을 출력할 때, 행과 컬럼이 모두 생략되지 않도록 하는 코드
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

df = pd.read_csv("DieCasting_Quality_Raw_Data.csv", header=[0,1])
df.head()

Process                                                                   \
       id Product_Type Shot Velocity_1 Velocity_2 Velocity_3 High_Velocity   
0       1            1    1      0.144      0.170      0.188         2.134   
1    1002            1    2      0.144      0.170      0.182         2.124   
2    2003            1    3      0.144      0.170      0.182         2.116   
3    3004            1    4      0.144      0.170      0.182         2.137   
4    4005            1    5      0.144      0.172      0.176         2.111   

                                                                        \
  Cylinder_Pressure Rapid_Rise_Time Biscuit_Thickness  Clamping_Force    
0               214           0.008                 10             258   
1               217           0.008                 11             257   
2               214           0.008                 11             257   
3               217           0.008                 11             257   
4               217           0.008                 12             257   

                                                                           \
  Cycle_Time  Pressure_Rise_Time Casting_Pressure Spray_Time Spray_1_Time   
0       20.7               0.044             1037        7.8          0.7   
1       20.7               0.044             1052        7.8          0.7   
2       20.8               0.041             1037        7.8          0.7   
3       20.7               0.043             1051        7.8          0.7   
4       20.7               0.042             1052        7.8          0.7   

                             Sensor                                \
  Spray_2_Time Melting_Furnace_Temp Air_Pressure Air_Pressure_Min   
0          0.8                695.0          6.3                3   
1          0.8                696.4          6.3                3   
2          0.8                696.4          6.3                3   
3          0.8                696.4          6.3                3   
4          0.8                697.9          6.4                3   

                                                                   \
  Air_Pressure_Max Coolant_Temp Coolant_Temp_Min Coolant_Temp_Max   
0                9         26.0               10               50   
1                9         26.1               10               50   
2                9         26.1               10               50   
3                9         26.1               10               50   
4                9         26.1               10               50   

                                                                   \
  Coolant_Pressure Factory_Temp Factory_Temp_Min Factory_Temp_Max   
0             2.71         32.9             18.0             22.0   
1             2.69         32.9             18.0             22.0   
2             2.69         32.9             18.0             22.0   
3             2.69         32.9             18.0             22.0   
4             2.69         32.9             18.0             22.0   

                                                                  Defects  \
  Factory_Humidity Factory_Humidity_Min Factory_Humidity_Max Short_Shot_1   
0             58.4                 18.0                 22.0            0   
1             58.2                 18.0                 22.0            0   
2             58.2                 18.0                 22.0            0   
3             58.2                 18.0                 22.0            0   
4             57.8                 18.0                 22.0            0   

                                                                   \
  Bubble_1 Exfoliation_1 Blow_Hole_1 Stain_1 Dent_1 Deformation_1   
0        0             0           0       0      0             0   
1        0             0           0       0      0             0   
2        0             0           0       0      0             0   
3        0             1           0       0      0        

# Defects 값을 0/1로 통일

In [13]:
# =========================
# 2) 메인 데이터 컬럼 구조 확인 (Process / Sensor / Defects)
# =========================
# MultiIndex 컬럼의 0레벨(최상위)의 컬럼들만 잘라서 가져옴
process_df = df.xs("Process", axis=1, level=0)
sensor_df  = df.xs("Sensor",  axis=1, level=0)
defects_df = df.xs("Defects", axis=1, level=0)

process_df.columns = process_df.columns.astype(str).str.strip()
sensor_df.columns  = sensor_df.columns.astype(str).str.strip()
defects_df.columns = defects_df.columns.astype(str).str.strip()

In [16]:
# 0보다 크면 불량(1)로 바꾸기
defects_df = defects_df.apply(pd.to_numeric, errors="coerce").fillna(0)
defects_df = (defects_df > 0).astype(int)
# defects_df = (defects_df > 0).astype(int) # 불량유형 지정 안됨

In [17]:
col_uniques = defects_df.apply(lambda s: sorted(s.dropna().unique()))
col_uniques

Short_Shot_1       [0, 1]
Bubble_1           [0, 1]
Exfoliation_1      [0, 1]
Blow_Hole_1        [0, 1]
Stain_1            [0, 1]
Dent_1             [0, 1]
Deformation_1      [0, 1]
Contamination_1    [0, 1]
Impurity_1         [0, 1]
Crack_1            [0, 1]
Scratch_1          [0, 1]
Buring_Mark_1      [0, 1]
Inclusions_1          [0]
Short_Shot_2       [0, 1]
Bubble_2           [0, 1]
Exfoliation_2      [0, 1]
Blow_Hole_2        [0, 1]
Stain_2               [0]
Dent_2             [0, 1]
Deformation_2      [0, 1]
Contamination_2    [0, 1]
Impurity_2         [0, 1]
Crack_2            [0, 1]
Scratch_2             [0]
Buring_Mark_2         [0]
Inclusions_2       [0, 1]
dtype: object

# Cavity 1/2 통합 (Defects 통합)

In [ ]:
# 원본 보관?
defects_raw_cavity = defects_df.copy()

# cavity 1/2 컬럼 분리
c1 = defects_df[[c for c in defects_df.columns if c.endswith("_1")]].copy()
c2 = defects_df[[c for c in defects_df.columns if c.endswith("_2")]].copy()

# 컬럼명 통일: Short_Shot_1 -> Short_Shot
c1.columns = [c.replace("_1", "") for c in c1.columns]
c2.columns = [c.replace("_2", "") for c in c2.columns]

# 제대로 분리되었는지 확인
print("c1 shape:", c1.shape)
print("c2 shape:", c2.shape)

c1 shape: (7535, 13)
c2 shape: (7535, 13)


In [21]:
all_defects = sorted(set(c1.columns) | set(c2.columns))

c1 = c1.reindex(columns=all_defects, fill_value=0)
c2 = c2.reindex(columns=all_defects, fill_value=0)

In [ ]:
# OR 방식으로 통합: 둘 중 하나라도 1이면 1
defects_merged = ((c1 + c2) > 0).astype(int)   # (c1 | c2) 도 가능
defects_df = defects_merged

print("통합 defects_df shape:", defects_df.shape)
print("값 종류:", np.unique(defects_df.to_numpy()))
defects_df.head(20)

통합 defects_df shape: (7535, 13)
값 종류: [0 1]


,Blow_Hole,Bubble,Buring_Mark,Contamination,Crack,Deformation,Dent,Exfoliation,Impurity,Inclusions,Scratch,Short_Shot,Stain
0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,1,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,1,0,0,0,0,0,0,0


# 각 불량 유형별로 묶음 (표면, 구조, 이물질 등)
분류 기준 정하고 하기

In [ ]:
#

# 결측값 처리(Sensor 중심) 

In [25]:
# =========================
# 결측치(NA) 구조 확인 (+ 컬럼별 비율)
# =========================
print("===== 결측치(NA) 확인 =====")

n_rows = len(df)

na_count = df.isna().sum().sort_values(ascending=False)
na_cols = na_count[na_count > 0]

# 그룹별 결측치 개수(총 결측 수) - 기존 그대로
print("그룹별 결측치 합(총 결측 수):")
print(pd.Series({
    "Process": process_df.isna().sum().sum(),
    "Sensor":  sensor_df.isna().sum().sum(),
    "Defects": defects_df.isna().sum().sum()
}))
print()

# 컬럼별 결측치 개수
print("결측치가 있는 컬럼만(개수):")
print(na_cols)
print()

# 컬럼별 결측치 비율(%): (결측 행 수 / 전체 행 수) * 100
print("결측치가 있는 컬럼만(비율 %):")
print(((na_cols / n_rows) * 100).round(4))
print()

===== 결측치(NA) 확인 =====
그룹별 결측치 합(총 결측 수):
Process      0
Sensor     540
Defects      0
dtype: int64

결측치가 있는 컬럼만(개수):
Sensor  Factory_Temp            90
        Factory_Humidity        90
        Factory_Temp_Max        90
        Factory_Humidity_Max    90
        Factory_Humidity_Min    90
        Factory_Temp_Min        90
dtype: int64

결측치가 있는 컬럼만(비율 %):
Sensor  Factory_Temp            1.1944
        Factory_Humidity        1.1944
        Factory_Temp_Max        1.1944
        Factory_Humidity_Max    1.1944
        Factory_Humidity_Min    1.1944
        Factory_Temp_Min        1.1944
dtype: float64



In [29]:
# Sensor에서 _MIN/_MAX로 끝나는 컬럼 찾아서 삭제
minmax_cols = [c for c in sensor_df.columns if c.endswith("_Min") or c.endswith("_Max")]
sensor_df_clean = sensor_df.drop(columns=minmax_cols)

print("삭제한 _MIN/_MAX 컬럼 수:", len(minmax_cols))
print("삭제 전 sensor_df shape:", sensor_df.shape)
print("삭제 후 sensor_df shape:", sensor_df_clean.shape)

삭제한 _MIN/_MAX 컬럼 수: 8
삭제 전 sensor_df shape: (7535, 14)
삭제 후 sensor_df shape: (7535, 6)


In [31]:
sensor_df_clean.head()

,Melting_Furnace_Temp,Air_Pressure,Coolant_Temp,Coolant_Pressure,Factory_Temp,Factory_Humidity
0,695.0,6.3,26.0,2.71,32.9,58.4
1,696.4,6.3,26.1,2.69,32.9,58.2
2,696.4,6.3,26.1,2.69,32.9,58.2
3,696.4,6.3,26.1,2.69,32.9,58.2
4,697.9,6.4,26.1,2.69,32.9,57.8


In [34]:
# =========================
# 결측치(NA) 구조 확인 (+ 컬럼별 비율)
# =========================
print("===== 결측치(NA) 확인 =====")

n_rows = len(sensor_df_clean)

na_count = sensor_df_clean.isna().sum().sort_values(ascending=False)
na_cols = na_count[na_count > 0]

# 컬럼별 결측치 개수
print("결측치가 있는 컬럼만(개수):")
print(na_cols)
print()


===== 결측치(NA) 확인 =====
결측치가 있는 컬럼만(개수):
Factory_Temp        90
Factory_Humidity    90
dtype: int64



In [38]:
sensor_df_clean[["Factory_Temp", "Factory_Humidity"]] = (
    sensor_df_clean[["Factory_Temp", "Factory_Humidity"]]
    .interpolate(method="linear", limit_direction="both")
)
print(sensor_df_clean.isna().sum())

Melting_Furnace_Temp    0
Air_Pressure            0
Coolant_Temp            0
Coolant_Pressure        0
Factory_Temp            0
Factory_Humidity        0
dtype: int64


In [39]:
process_df.head()

,id,Product_Type,Shot,Velocity_1,Velocity_2,Velocity_3,High_Velocity,Cylinder_Pressure,Rapid_Rise_Time,Biscuit_Thickness,Clamping_Force,Cycle_Time,Pressure_Rise_Time,Casting_Pressure,Spray_Time,Spray_1_Time,Spray_2_Time
0,1,1,1,0.144,0.170,0.188,2.134,214,0.008,10,258,20.7,0.044,1037,7.8,0.7,0.8
1,1002,1,2,0.144,0.170,0.182,2.124,217,0.008,11,257,20.7,0.044,1052,7.8,0.7,0.8
2,2003,1,3,0.144,0.170,0.182,2.116,214,0.008,11,257,20.8,0.041,1037,7.8,0.7,0.8
3,3004,1,4,0.144,0.170,0.182,2.137,217,0.008,11,257,20.7,0.043,1051,7.8,0.7,0.8
4,4005,1,5,0.144,0.172,0.176,2.111,217,0.008,12,257,20.7,0.042,1052,7.8,0.7,0.8


# 식별자/그룹 구분 